# 03 — Results figures

Generates all publication-quality figures from the Monte Carlo grid results.

**Input:** `results/grid_main.parquet` (or any parquet produced by `02_power_grid.py`)  
**Output:** figures saved to `results/figures/`

Figures:
1. FDR empirique vs α nominal — tous algos, régime modéré
2. Puissance vs fréquence — tous algos, α=0.05, régime modéré
3. Heatmap puissance par (algo × jump regime) — dt=5s, α=0.05
4. Effet wealth_fraction sur power et α-death (GAI-family seulement)
5. Wall-clock par algo (tableau)

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# config
PARQUET = Path("../results/grid_main.parquet")
OUTDIR  = Path("../results/figures")
OUTDIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PARQUET)
print(f"Loaded {len(df):,} rows  |  columns: {list(df.columns)}")
print(df.dtypes)
df.head(3)

In [ ]:
# helper: fdr validity table (criterion: fdr <= alpha + 2*se)
def fdr_validity_table(df: pd.DataFrame) -> pd.DataFrame:
    """Group by (algo, alpha) → mean FDR, SE, criterion pass/fail."""
    g = df.groupby(["algo", "alpha"])
    out = g.agg(
        fdr_mean=("fdp", "mean"),
        fdr_se=("fdp", lambda x: x.std() / np.sqrt(len(x))),
        n_runs=("fdp", "count"),
    ).reset_index()
    out["criterion"] = out["fdr_mean"] <= (out["alpha"] + 2 * out["fdr_se"])
    return out.sort_values(["alpha", "algo"])

validity = fdr_validity_table(df)
n_violations = (~validity["criterion"]).sum()
print(f"\nFDR validity: {n_violations} violations out of {len(validity)} (algo, alpha) pairs")
if n_violations > 0:
    print(validity[~validity["criterion"]].to_string())
else:
    print("All algorithms satisfy FDR ≤ alpha + 2·SE  ✓")
validity

In [ ]:
# figure 1 — fdr empirique vs α nominal (régime modéré, toutes fréquences)
REGIME   = "moderate"
ALGOS    = ["ebh", "bh_lm", "bh_bns", "elond", "elord", "esaffron", "stopped_ebh"]
COLORS   = plt.rcParams["axes.prop_cycle"].by_key()["color"]
ALGO_COLOR = {a: COLORS[i % len(COLORS)] for i, a in enumerate(ALGOS)}
MARKERS  = ["o", "s", "^", "D", "v", "P", "*"]
ALGO_MK  = {a: MARKERS[i] for i, a in enumerate(ALGOS)}

sub = df[df["regime"] == REGIME]
alphas = sorted(sub["alpha"].unique())

fig, ax = plt.subplots(figsize=(7, 5))
for algo in ALGOS:
    pts = sub[sub["algo"] == algo].groupby("alpha")["fdp"].agg(["mean", "sem"]).reset_index()
    ax.errorbar(
        pts["alpha"], pts["mean"], yerr=2 * pts["sem"],
        label=algo, color=ALGO_COLOR[algo], marker=ALGO_MK[algo],
        capsize=4, linewidth=1.5,
    )

# Diagonal: FDR = alpha (perfect control)
alpha_range = np.linspace(0, max(alphas) * 1.1, 50)
ax.plot(alpha_range, alpha_range, "k--", linewidth=0.8, label="FDR = α (perfect)")

ax.set_xlabel("α nominal", fontsize=12)
ax.set_ylabel("FDR empirique (mean ± 2·SE)", fontsize=12)
ax.set_title(f"FDR control — regime={REGIME}, all frequencies", fontsize=13)
ax.legend(fontsize=9, ncol=2)
ax.set_xlim(0, max(alphas) * 1.15)
ax.set_ylim(-0.01, max(alphas) * 1.5)
fig.tight_layout()
fig.savefig(OUTDIR / "fig1_fdr_vs_alpha.pdf", dpi=150)
plt.show()
print("Saved fig1_fdr_vs_alpha.pdf")

In [ ]:
# figure 2 — puissance vs fréquence (α=0.05, régime modéré)
ALPHA = 0.05

sub2 = df[(df["regime"] == REGIME) & (df["alpha"] == ALPHA)]
freqs = sorted(sub2["dt_seconds"].unique())

fig, ax = plt.subplots(figsize=(8, 5))
for algo in ALGOS:
    pts = (
        sub2[sub2["algo"] == algo]
        .groupby("dt_seconds")["power"]
        .agg(["mean", "sem"])
        .reset_index()
    )
    ax.errorbar(
        pts["dt_seconds"], pts["mean"], yerr=2 * pts["sem"],
        label=algo, color=ALGO_COLOR[algo], marker=ALGO_MK[algo],
        capsize=4, linewidth=1.5,
    )

ax.set_xscale("log")
ax.set_xlabel("Fréquence d'échantillonnage (secondes/tick, log)", fontsize=12)
ax.set_ylabel("Puissance empirique (mean ± 2·SE)", fontsize=12)
ax.set_title(f"Puissance vs fréquence — regime={REGIME}, α={ALPHA}", fontsize=13)
ax.set_xticks(freqs)
ax.set_xticklabels([f"{int(f)}s" for f in freqs])
ax.legend(fontsize=9, ncol=2)
ax.set_ylim(-0.05, 1.05)
fig.tight_layout()
fig.savefig(OUTDIR / "fig2_power_vs_freq.pdf", dpi=150)
plt.show()
print("Saved fig2_power_vs_freq.pdf")

In [ ]:
# figure 3 — heatmap puissance par (algo × regime) — dt=5s, α=0.05
DT_FOCUS = 5.0

sub3 = df[(df["dt_seconds"] == DT_FOCUS) & (df["alpha"] == ALPHA)]
regimes = sorted(sub3["regime"].unique())

power_matrix = pd.pivot_table(
    sub3, values="power", index="algo", columns="regime", aggfunc="mean"
).reindex(index=ALGOS, columns=regimes)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(power_matrix.values, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
plt.colorbar(im, ax=ax, label="Puissance empirique")

ax.set_xticks(range(len(regimes)))
ax.set_xticklabels(regimes, fontsize=11)
ax.set_yticks(range(len(ALGOS)))
ax.set_yticklabels(ALGOS, fontsize=11)

# Annotate cells
for i, algo in enumerate(ALGOS):
    for j, regime in enumerate(regimes):
        val = power_matrix.loc[algo, regime]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=9, color="black" if 0.2 < val < 0.8 else "white")

ax.set_title(f"Puissance par (algo, regime) — dt={int(DT_FOCUS)}s, α={ALPHA}", fontsize=13)
fig.tight_layout()
fig.savefig(OUTDIR / "fig3_power_heatmap.pdf", dpi=150)
plt.show()
print("Saved fig3_power_heatmap.pdf")

In [ ]:
# figure 4 — effet wealth_fraction sur power et fdr (gai-family)
GAI_ALGOS = ["elord", "esaffron"]

# The grid stores w1 implicitly in the FDRConfig; the runner stores it in the
# output row. Check if 'w1' column exists.
if "w1" not in df.columns:
    print("Column 'w1' not in parquet — runner must emit it for this figure.")
    print("Skipping Figure 4.")
else:
    sub4 = df[
        (df["algo"].isin(GAI_ALGOS))
        & (df["regime"] == REGIME)
        & (df["dt_seconds"] == DT_FOCUS)
        & (df["alpha"] == ALPHA)
    ]
    w1_vals = sorted(sub4["w1"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, metric, label in zip(axes, ["power", "fdp"], ["Puissance", "FDR empirique"]):
        for algo in GAI_ALGOS:
            pts = (
                sub4[sub4["algo"] == algo]
                .groupby("w1")[metric]
                .agg(["mean", "sem"])
                .reset_index()
            )
            ax.errorbar(
                pts["w1"], pts["mean"], yerr=2 * pts["sem"],
                label=algo, marker="o", capsize=4,
            )
        if metric == "fdp":
            ax.axhline(ALPHA, color="red", linestyle="--", linewidth=0.8, label="α target")
        ax.set_xlabel("w1 (wealth fraction)", fontsize=12)
        ax.set_ylabel(label, fontsize=12)
        ax.set_title(f"{label} vs w1", fontsize=12)
        ax.legend()
    fig.suptitle(f"Effet wealth_fraction — GAI algos, {REGIME}, dt={int(DT_FOCUS)}s, α={ALPHA}",
                 fontsize=13)
    fig.tight_layout()
    fig.savefig(OUTDIR / "fig4_wealth_effect.pdf", dpi=150)
    plt.show()
    print("Saved fig4_wealth_effect.pdf")

In [ ]:
# figure 5 — wall-clock par algo (tableau)
wc = (
    df.groupby("algo")["wall_clock"]
    .agg(["mean", "median", "std", "max"])
    .round(4)
    .sort_values("mean")
    .rename(columns={"mean": "mean(s)", "median": "median(s)", "std": "std(s)", "max": "max(s)"})
)
print("Wall-clock per run (seconds)")
print(wc.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(wc.index, wc["mean(s)"], yerr=wc["std(s)"], capsize=5,
       color=[ALGO_COLOR.get(a, "steelblue") for a in wc.index])
ax.set_xlabel("Algorithme")
ax.set_ylabel("Temps moyen (s)")
ax.set_title("Wall-clock par algorithme (mean ± std)")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(OUTDIR / "fig5_wallclock.pdf", dpi=150)
plt.show()
print("Saved fig5_wallclock.pdf")

In [ ]:
# tableau de ranking final par cellule (freq × regime × alpha)
ranking = (
    df.groupby(["algo", "dt_seconds", "regime", "alpha"])
    .agg(
        fdr=("fdp", "mean"),
        fdr_se=("fdp", lambda x: x.std() / np.sqrt(len(x))),
        power=("power", "mean"),
        f1=("f1", "mean"),
        rej_rate=("rej_rate", "mean"),
        n_runs=("fdp", "count"),
    )
    .reset_index()
)
ranking["fdr_ok"] = ranking["fdr"] <= (ranking["alpha"] + 2 * ranking["fdr_se"])
ranking = ranking.round(4)

# Save ranking CSV
ranking.to_csv(OUTDIR / "ranking_table.csv", index=False)
print(f"Ranking table: {len(ranking)} rows saved to results/figures/ranking_table.csv")

# Pivot: power at dt=5s, alpha=0.05
focus = ranking[(ranking["dt_seconds"] == 5.0) & (ranking["alpha"] == 0.05)]
pivot = focus.pivot_table(index="algo", columns="regime", values="power").round(3)
print("\nPower ranking — dt=5s, α=0.05")
print(pivot.to_string())